# EnergyCast V1 — Step 12: Model Benchmarking, Statistical Significance & Master Evaluation

This notebook conducts the rigorous, unified benchmarking of all forecasting models developed across Tiers 1, 2, and 3:
- **Tier 1 (Baselines)**: Persistence (1h), Daily Seasonal Naive (24h), Weekly Seasonal Naive (168h)
- **Tier 2 (Linear / Regularized)**: Ordinary Least Squares (OLS), Ridge Regression ($\alpha=50.0$)
- **Tier 3 (GBDT Ensembles)**: XGBoost Regressor, LightGBM Regressor (Champion)

### Key Objectives:
1. **Master Model Leaderboard**: Multi-metric evaluation (MAE, RMSE, sMAPE, $R^2$) across Validation (2024) and Test (2025) splits.
2. **Diebold-Mariano (1995) Hypothesis Testing**: Verify if LightGBM's superior performance is statistically significant under Newey-West HAC standard errors.
3. **Segmented Performance**: Evaluate Peak (18:00–22:00 IST), Off-Peak, and Extreme Price Spike ($> ₹7,000/\text{MWh}$) subsets.
4. **Residual Diagnostics**: Analyze error distribution kurtosis, skewness, and Jarque-Bera normality tests.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.benchmark_models import load_and_merge_predictions, diebold_mariano_test, compute_metrics

print("Environment initialized.")

## 1. Load Aligned Multi-Model Predictions

In [ ]:
df_test = load_and_merge_predictions("test")
df_val = load_and_merge_predictions("val")

print(f"Test set: {len(df_test):,} hours ({df_test['timestamp'].min()} to {df_test['timestamp'].max()})")
print(f"Validation set: {len(df_val):,} hours ({df_val['timestamp'].min()} to {df_val['timestamp'].max()})")
df_test.head()

## 2. Master Model Leaderboard (Validation & Test Sets)

In [ ]:
leaderboard_path = PROJECT_ROOT / "outputs" / "tables" / "master_model_leaderboard.csv"
df_leaderboard = pd.read_csv(leaderboard_path)
df_leaderboard[["Model", "Test_MAE", "Test_RMSE", "Test_sMAPE", "Test_R2", "MAE_Reduction_vs_SNaive_%", "Test_Peak_MAE", "Test_Spike_MAE"]]

## 3. Diebold-Mariano (1995) Statistical Significance Testing

In [ ]:
dm_path = PROJECT_ROOT / "outputs" / "tables" / "diebold_mariano_pairwise_tests.csv"
df_dm = pd.read_csv(dm_path)

# Filter for champion comparisons
champ_dm = df_dm[(df_dm["Model_1"] == "LightGBM Regressor (Champion)") | (df_dm["Model_2"] == "LightGBM Regressor (Champion)")]
champ_dm[["Model_1", "Model_2", "DM_Statistic", "p_value_two_sided", "Significance_at_1%", "Superior_Model"]]

## 4. Residual Error Diagnostics

In [ ]:
res_path = PROJECT_ROOT / "outputs" / "tables" / "residual_error_diagnostics.csv"
df_res = pd.read_csv(res_path)
df_res[["Model", "Mean_Error (Bias)", "Std_Error", "Skewness", "Kurtosis", "Jarque_Bera_Stat", "JB_p_value"]]

## 5. Visualizing Benchmark Comparisons

In [ ]:
from IPython.display import Image, display

fig_dir = PROJECT_ROOT / "outputs" / "figures" / "benchmarks"
display(Image(filename=str(fig_dir / "fig_bench01_master_leaderboard_bar.png")))
display(Image(filename=str(fig_dir / "fig_bench02_diebold_mariano_heatmap.png")))
display(Image(filename=str(fig_dir / "fig_bench03_hourly_mae_profile_all_models.png")))
display(Image(filename=str(fig_dir / "fig_bench04_cumulative_absolute_loss.png")))
display(Image(filename=str(fig_dir / "fig_bench05_error_residuals_distribution.png")))